In [3]:
%pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 26.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 105.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.6 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: numpy━━━━━━━━━━━━━━━━━━━ 0/5 [polars-runtime-32]
    Found existing installation: numpy 2.2.6 0/5 [polars-runtime-32]
    Uninstalling numpy-2.2.6:m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [numpy]
      Successfully uninstalled numpy-2.2.6━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [numpy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ultralytics] [ultralytics]thop]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda11x 11.0.0 requires numpy<1.26,>=

In [4]:
import os

from ultralytics import YOLO
from pathlib import Path
import shutil
import random
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import cv2
import warnings

In [2]:
# Путь к исходным данным
DATASET_PATH = Path("/home/jupyter/project/tiny")

# Пути для YOLO-датасета
YOLO_DATASET = Path("/home/jupyter/project/yolo_dataset")
TRAIN_PATH = YOLO_DATASET / "train"
VAL_PATH   = YOLO_DATASET / "val"

# Параметры разбивки
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.20

# Параметры обучения
EPOCHS      = 30
IMG_SIZE    = 224
BATCH_SIZE  = 16
MODEL_NAME  = "yolov8n-cls.pt"   # nano-классификатор (быстрый)

# Все классы (28 штук)
CLASSES = sorted([
    d.name for d in DATASET_PATH.iterdir() 
    if d.is_dir()
])

print(f"✅ Найдено классов: {len(CLASSES)}")
print(f"📂 Классы: {CLASSES}")


✅ Найдено классов: 28
📂 Классы: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'nothing', 'space']


In [5]:
def prepare_yolo_dataset(
    source_path: Path,
    target_path: Path,
    classes: list,
    train_r=0.80,
    val_r=0.20,
    seed=42
):

    random.seed(seed)
    
    # Удаляем старую папку (если есть)
    if target_path.exists():
        shutil.rmtree(target_path)
        print("🗑️  Старый yolo_dataset удалён")
    
    stats = {"train": 0, "val": 0, "test": 0}
    
    for cls in classes:
        src_cls = source_path / cls
        all_imgs = (
            list(src_cls.glob("*.jpg"))  +
            list(src_cls.glob("*.jpeg")) +
            list(src_cls.glob("*.png"))  +
            list(src_cls.glob("*.bmp"))
        )
        
        if not all_imgs:
            print(f"⚠️  Класс '{cls}' — нет изображений, пропускаем")
            continue
        
        random.shuffle(all_imgs)
        
        n        = len(all_imgs)
        n_train  = int(n * train_r)
        n_val    = int(n * val_r)
        
        splits = {
            "train" : all_imgs[:n_train],
            "val"   : all_imgs[n_train : n_train + n_val],
            "test"  : all_imgs[n_train + n_val:]
        }
        
        for split, imgs in splits.items():
            dst = target_path / split / cls
            dst.mkdir(parents=True, exist_ok=True)
            for img_path in imgs:
                shutil.copy2(img_path, dst / img_path.name)
            stats[split] += len(imgs)
    
    return stats


print("⏳ Подготовка датасета...")
stats = prepare_yolo_dataset(
    DATASET_PATH, YOLO_DATASET, CLASSES,
    TRAIN_RATIO, VAL_RATIO
)

print("\n✅ Датасет подготовлен!")
print(f"   Train : {stats['train']:>6} изображений")
print(f"   Val   : {stats['val']:>6} изображений")
print(f"   Test  : {stats['test']:>6} изображений")
print(f"   Total : {sum(stats.values()):>6} изображений")

# Проверяем структуру
print("\n📁 Структура yolo_dataset:")
for split in ["train", "val", "test"]:
    split_path = YOLO_DATASET / split
    n_classes = len(list(split_path.iterdir()))
    print(f"   {split}/  → {n_classes} классов")

⏳ Подготовка датасета...

✅ Датасет подготовлен!
   Train :  11200 изображений
   Val   :   2800 изображений
   Test  :      0 изображений
   Total :  14000 изображений

📁 Структура yolo_dataset:
   train/  → 28 классов
   val/  → 28 классов
   test/  → 28 классов


In [6]:
DATASET = Path("/home/jupyter/project/yolo_dataset")

model = YOLO("yolov8n-cls.pt")

results = model.train(
    data       = str(DATASET.resolve()),
    epochs     = 30,
    imgsz      = 224,
    batch      = 8,
    amp        = False,
    workers    = 4,
    patience   = 10,
    lr0        = 0.001,
    dropout    = 0.3,
    augment    = True,
    name       = "asl_yolo_cls",
    verbose    = True,
)

print(f"Top-1: {results.top1:.4f}")
print(f"Top-5: {results.top5:.4f}")

New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.41 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jupyter/project/yolo_dataset, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.3, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0

In [9]:
RUN_PATH = Path("/home/jupyter/project/runs/classify/asl_yolo_cls-2")
import pandas as pd
import numpy as np

csv_path = RUN_PATH / "results.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

print("📊 Все колонки в results.csv:")
print(f"   {list(df.columns)}")
print(f"\n   Эпох записано: {len(df)}")
print("\n" + "=" * 55)
print(df.to_string(index=False))

📊 Все колонки в results.csv:
   ['epoch', 'time', 'train/loss', 'metrics/accuracy_top1', 'metrics/accuracy_top5', 'val/loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']

   Эпох записано: 30

 epoch      time  train/loss  metrics/accuracy_top1  metrics/accuracy_top5  val/loss   lr/pg0   lr/pg1   lr/pg2
     1   71.9368     2.92088                0.70964                0.94679   1.37615 0.000104 0.000104 0.000104
     2  144.2260     1.30866                0.89893                0.98036   0.40157 0.000202 0.000202 0.000202
     3  212.4130     0.76774                0.93929                0.99179   0.22361 0.000292 0.000292 0.000292
     4  281.5220     0.58106                0.95071                0.99607   0.16738 0.000282 0.000282 0.000282
     5  351.3530     0.47733                0.95821                0.99500   0.12840 0.000272 0.000272 0.000272
     6  420.4770     0.42206                0.96357                0.99679   0.11524 0.000261 0.000261 0.000261
     7  490.0640     0.37086         

In [10]:
best_idx     = df["metrics/accuracy_top1"].idxmax()
best_epoch   = df.loc[best_idx, "epoch"]
best_top1    = df.loc[best_idx, "metrics/accuracy_top1"]
best_top5    = df.loc[best_idx, "metrics/accuracy_top5"]
last_row     = df.iloc[-1]

print("=" * 55)
print("        ИТОГОВЫЕ МЕТРИКИ ОБУЧЕНИЯ")
print("=" * 55)

print(f"""
  Всего эпох обучения  : {int(df['epoch'].max())}
  
  ┌─ Лучшая эпоха ──────────────────────────┐
  │  Эпоха            : {int(best_epoch)}
  │  Top-1 Accuracy   : {best_top1:.4f}  ({best_top1*100:.2f}%)
  │  Top-5 Accuracy   : {best_top5:.4f}  ({best_top5*100:.2f}%)
  └──────────────────────────────────────────┘
  
  ┌─ Последняя эпоха ───────────────────────┐
  │  Эпоха            : {int(last_row['epoch'])}
  │  Train Loss       : {last_row['train/loss']:.4f}
  │  Val Loss         : {last_row['val/loss']:.4f}
  │  Top-1 Accuracy   : {last_row['metrics/accuracy_top1']:.4f}  ({last_row['metrics/accuracy_top1']*100:.2f}%)
  │  Top-5 Accuracy   : {last_row['metrics/accuracy_top5']:.4f}  ({last_row['metrics/accuracy_top5']*100:.2f}%)
  └──────────────────────────────────────────┘
""")

        ИТОГОВЫЕ МЕТРИКИ ОБУЧЕНИЯ

  Всего эпох обучения  : 30
  
  ┌─ Лучшая эпоха ──────────────────────────┐
  │  Эпоха            : 30
  │  Top-1 Accuracy   : 0.9854  (98.54%)
  │  Top-5 Accuracy   : 0.9986  (99.86%)
  └──────────────────────────────────────────┘
  
  ┌─ Последняя эпоха ───────────────────────┐
  │  Эпоха            : 30
  │  Train Loss       : 0.1666
  │  Val Loss         : 0.0472
  │  Top-1 Accuracy   : 0.9854  (98.54%)
  │  Top-5 Accuracy   : 0.9986  (99.86%)
  └──────────────────────────────────────────┘



In [11]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

epochs = df["epoch"]

<Figure size 1800x1200 with 0 Axes>

In [12]:
fig.suptitle(
    f"YOLO ASL Classifier — Результаты обучения\n"
    f"Лучшая точность: {best_top1:.2%} (эпоха {int(best_epoch)})",
    fontsize=15, fontweight="bold", y=1.01
)

plt.savefig(
    str(RUN_PATH / "training_curves.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()
print("✅ График сохранён: training_curves.png")

<Figure size 640x480 with 0 Axes>

✅ График сохранён: training_curves.png


In [15]:
model.save('/home/jupyter/project/Yolo/sign_language_yolo.keras')
print('✅ Модель сохранена!')

✅ Модель сохранена!
